## Necessary initialization

In [1]:
from pathlib import Path
import sys, shutil

import pandas as pd
from loguru import logger
from datetime import datetime, timedelta

# Relative imports
from collector import YahooCollectorUS, Run


# Append dump_bin to sys.path
CUR_DIR = Path().resolve()
sys.path.append(str(CUR_DIR.parent.parent))

from dump_bin import DumpDataAll, DumpDataUpdate, DumpDataFix


# Target database
qlib_dir = Path().resolve().parent.parent.parent.joinpath("qlib_data_US")
qlib_dir

WindowsPath('D:/Projects/Investing/QuantitativeML_Investing/Database/qlib_data_US')

## Rewrite YahooCollectorUS.get_instrument_list
To pull only selected symbols

In [2]:
# Rewrite YahooCollectorUS.get_instrument_list
tickers_sp500 = pd.read_csv(qlib_dir.joinpath("instruments/sp500_only_new.txt"), sep='\t', header=None, names=['symbol', 'start_date', 'end_date'])
tickers_sp400 = pd.read_csv(qlib_dir.joinpath("instruments/sp400_only_new.txt"), sep='\t', header=None, names=['symbol', 'start_date', 'end_date'])


tickers = (
    ['^GSPC']  # '^MID', '^SP600'
    + tickers_sp500['symbol'].tolist()
    + tickers_sp400['symbol'].tolist()
)
tickers = [ticker.replace('.', '-') for ticker in tickers]


def f(self):

    logger.info("get US stock symbols......")

    symbols = tickers

    logger.info(f"get {len(symbols)} symbols.")
    return symbols


YahooCollectorUS.get_instrument_list = f


## RUN

In [3]:
r = Run(region="US", interval="1d")

### Download data

In [4]:
r.download_data(
    max_collector_count=2,
    delay=0.5,
    start='1996-01-01',
    end=(datetime.today() + timedelta(days=1)).strftime('%Y-%m-%d'),
)

2024-07-30 09:56:12.698 | INFO     | __main__:f:22 - get US stock symbols......
2024-07-30 09:56:12.698 | INFO     | __main__:f:26 - get 905 symbols.
2024-07-30 09:56:12.761 | INFO     | data_collector.base:collector_data:202 - start collector data......
2024-07-30 09:56:12.776 | INFO     | data_collector.base:collector_data:207 - getting data: 1
100%|██████████| 905/905 [22:42<00:00,  1.51s/it]


[]


2024-07-30 10:18:55.429 | INFO     | data_collector.base:_collector:195 - error symbol nums: 0
2024-07-30 10:18:55.429 | INFO     | data_collector.base:_collector:196 - current get symbol nums: 905
2024-07-30 10:18:55.429 | INFO     | data_collector.base:collector_data:209 - 1 finish.
2024-07-30 10:18:55.429 | INFO     | data_collector.base:collector_data:216 - total 905, error: 0


DONE


### Normalize data

In [5]:
r.normalize_data()

2024-07-30 10:18:55.446 | INFO     | data_collector.utils:get_calendar_list:68 - get calendar list: US_ALL......


                          open         high          low        close  \
symbol date                                                             
^GSPC  1942-01-02     8.890000     8.890000     8.890000     8.890000   
       1942-01-05     9.090000     9.090000     9.090000     9.090000   
       1942-01-06     9.050000     9.050000     9.050000     9.050000   
       1942-01-07     9.000000     9.000000     9.000000     9.000000   
       1942-01-08     8.880000     8.880000     8.880000     8.880000   
...                        ...          ...          ...          ...   
       2024-07-23  5565.299805  5585.339844  5550.899902  5555.740234   
       2024-07-24  5505.839844  5508.040039  5419.979980  5427.129883   
       2024-07-25  5428.700195  5491.589844  5390.950195  5399.220215   
       2024-07-26  5433.669922  5488.319824  5430.700195  5459.100098   
       2024-07-29  5476.549805  5487.740234  5444.439941  5463.540039   

                       volume     adjclose  
symbo

2024-07-30 10:18:57.411 | INFO     | data_collector.utils:get_calendar_list:108 - end of get calendar list: US_ALL.
2024-07-30 10:18:57.411 | INFO     | data_collector.base:normalize:313 - normalize data......
100%|██████████| 905/905 [03:17<00:00,  4.58it/s]


DONE


## DUMP DATA

In [6]:
_dump = DumpDataAll(
    csv_path=Path().resolve().joinpath("source"),
    qlib_dir=qlib_dir,
    exclude_fields="symbol,date",
    max_workers=4,
)
_dump.dump()

2024-07-30 10:30:33.189 | INFO     | dump_bin:_get_all_date:274 - start get all date......
100%|██████████| 905/905 [00:22<00:00, 40.42it/s]
2024-07-30 10:30:55.593 | INFO     | dump_bin:_get_all_date:293 - end of get all date.

2024-07-30 10:30:55.593 | INFO     | dump_bin:_dump_calendars:296 - start dump calendars......
2024-07-30 10:30:55.826 | INFO     | dump_bin:_dump_calendars:299 - end of calendars dump.

2024-07-30 10:30:55.826 | INFO     | dump_bin:_dump_instruments:302 - start dump instruments......
2024-07-30 10:30:55.826 | INFO     | dump_bin:_dump_instruments:304 - end of instruments dump.

2024-07-30 10:30:55.826 | INFO     | dump_bin:_dump_features:307 - start dump features......
100%|██████████| 905/905 [00:30<00:00, 29.41it/s]
2024-07-30 10:31:26.594 | INFO     | dump_bin:_dump_features:314 - end of features dump.



## Clear

In [2]:
shutil.rmtree(Path().resolve().joinpath("source"))
shutil.rmtree(Path().resolve().joinpath("normalize"))